# Study 2 — Does LoRA's low rank act as regularisation when labels are scarce?

Model and corpus fixed (Qwen2.5-0.5B, Financial PhraseBank); **only the amount of
labelled training data varies**.

| | |
|---|---|
| Training sizes | 100, 250, 500, 1000, 2417 (stratified) |
| Methods | LoRA r = 1, 2, 4, 8, 16 · full fine-tuning · full + heavy weight decay |
| Seeds | 42, 43, 44 |
| Budget | 5 epochs, floor of 150 updates, early stopping on validation macro-F1 |

**H1** low-rank helps more when supervision is scarce ·
**H2** rank sensitivity fades as data grows ·
**H3** the benefit is the low-rank *structure*, not just fewer parameters (`fullreg`)

105 runs, ~3 hours, designed to be left running unattended.

---

### Order

1. **§1 Install** — restart the runtime if told to
2. **§2 Load code** — three cells; re-run after any restart
3. **§3 Mount Drive** — one click from you; results survive a runtime wipe
4. **§4 Full run** — start it and walk away
5. **§5 Summary** — when you come back

Before §4: `caffeinate -d` in Terminal, tab in the foreground, on power.


## 1. Install

Colab ships a `torchao` newer `peft` refuses to load. It must go, **and the runtime must restart** for pip's metadata change to register.

In [ ]:
!pip uninstall -y torchao
!pip install -q "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" scikit-learn psutil

import importlib.util, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime to T4!")
still = importlib.util.find_spec("torchao") is not None
print("torchao still present:", still)
print(">>> RESTART THE RUNTIME NOW (Runtime > Restart session), then run \u00a72"
      if still else ">>> OK, continue to \u00a72")


## 2. Load the code

Run all three after any restart.

In [ ]:
%%writefile peft_study.py
"""
Route B - the same study on a real pretrained LLM and a real dataset
===================================================================
Run this in Google Colab (Runtime -> Change runtime type -> T4 GPU). Results are
written to colab_results.json after every single run, and main() skips anything
already finished - so if the session disconnects, just run the cell again.
It reproduces the comparison in the report on:

    model    Qwen/Qwen2.5-0.5B          (a real pretrained decoder-only LLM)
    data     financial_phrasebank       (sentences_75agree, 3-way sentiment)
    methods  LoRA (r = 1, 2, 4, 8, 16), full fine-tuning, linear probe

Total runtime on a free T4: roughly 25-40 minutes for the whole grid.

Paste the printed RESULTS TABLE back into the chat and the numbers will replace
the small-scale results in the report.

--------------------------------------------------------------------------
Cell 1 - install (run once, then restart the runtime if Colab asks)
--------------------------------------------------------------------------
!pip install -q "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" scikit-learn

--------------------------------------------------------------------------
Cell 2 - paste everything below
--------------------------------------------------------------------------
"""

import gc
import json
import time

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding)

MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
MAX_LEN = 96
BATCH = 16
EPOCHS = 3
SEEDS = [42, 43, 44]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Mixed precision. The T4 is a Turing card: it has fast fp16 tensor cores but no
# usable bf16, so we use fp16 + GradScaler. Master weights stay in fp32; only the
# forward/backward maths runs in half precision, which is numerically safe here
# and roughly halves the time per run.
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.float16


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

LABEL_NAMES = ['negative', 'neutral', 'positive']       # -> 0, 1, 2
AGREEMENT_FILE = 'Sentences_75Agree.txt'


def _parse_phrasebank_txt(raw_bytes):
    """The corpus is plain text, one 'sentence@label' pair per line, latin-1."""
    from datasets import ClassLabel, Dataset, Features, Value

    sentences, labels = [], []
    for line in raw_bytes.decode('latin-1').splitlines():
        line = line.strip()
        if not line or '@' not in line:
            continue
        sent, _, lab = line.rpartition('@')
        lab = lab.strip().lower()
        if lab not in LABEL_NAMES:
            continue
        sentences.append(sent.strip())
        labels.append(LABEL_NAMES.index(lab))

    if not sentences:
        raise ValueError('parsed zero sentences from the corpus file')

    features = Features({'sentence': Value('string'),
                         'labels': ClassLabel(names=LABEL_NAMES)})
    return Dataset.from_dict({'sentence': sentences, 'labels': labels},
                             features=features)


def load_phrasebank():
    """Financial PhraseBank, 75% annotator agreement. Labels: 0 neg, 1 neu, 2 pos.

    `datasets` 4.x removed support for loading scripts, which is exactly how this
    corpus used to be served, so `load_dataset("financial_phrasebank", ...)` now
    fails. The raw archive is still hosted in the same repository, so the primary
    path here downloads that file and parses it directly - no script machinery
    involved. Mirrors that were re-uploaded in a modern format are tried after,
    and a locally uploaded file last.
    """
    errors = []

    # 1) Canonical data, fetched as a plain file from the original repo.
    try:
        import io
        import zipfile

        from huggingface_hub import hf_hub_download

        path = hf_hub_download(repo_id='takala/financial_phrasebank',
                               filename='data/FinancialPhraseBank-v1.0.zip',
                               repo_type='dataset')
        with zipfile.ZipFile(path) as z:
            name = next(n for n in z.namelist() if n.endswith(AGREEMENT_FILE))
            ds = _parse_phrasebank_txt(z.read(name))
        print(f'loaded {len(ds)} sentences from takala/financial_phrasebank '
              f'({AGREEMENT_FILE})')
        return ds
    except Exception as e:                                        # noqa: BLE001
        errors.append(f'  original archive: {type(e).__name__}: {e}')

    # 2) Community mirrors already converted to a modern (script-free) format.
    for repo, config in [('FinanceMTEB/financial_phrasebank', None),
                         ('dohonba/financial_phrasebank', 'sentences_75agree'),
                         ('dohonba/financial_phrasebank', None)]:
        try:
            ds = load_dataset(repo, config) if config else load_dataset(repo)
            split = ds['train'] if 'train' in ds else ds[list(ds.keys())[0]]
            print(f'loaded mirror {repo} ({len(split)} rows)')
            return split
        except Exception as e:                                    # noqa: BLE001
            errors.append(f'  {repo} [{config}]: {type(e).__name__}: {e}')

    # 3) A file the user uploaded into the Colab session.
    import os
    for cand in (AGREEMENT_FILE, f'/content/{AGREEMENT_FILE}'):
        if os.path.exists(cand):
            with open(cand, 'rb') as fh:
                ds = _parse_phrasebank_txt(fh.read())
            print(f'loaded {len(ds)} sentences from uploaded {cand}')
            return ds

    raise RuntimeError(
        'Could not load Financial PhraseBank. Attempts:\n' + '\n'.join(errors)
        + f'\n\nManual fallback: download FinancialPhraseBank-v1.0.zip from\n'
          f'  https://huggingface.co/datasets/takala/financial_phrasebank/'
          f'blob/main/data/FinancialPhraseBank-v1.0.zip\n'
          f'unzip it, and upload {AGREEMENT_FILE} into the Colab file browser '
          f'(folder icon on the left), then re-run this cell.')


def get_data(tokenizer, seed=42):
    ds = load_phrasebank()

    # Mirrors use different column names; normalise to sentence / labels.
    for src in ('label', 'gold_label', 'target'):
        if src in ds.column_names and 'labels' not in ds.column_names:
            ds = ds.rename_column(src, 'labels')
    for src in ('text', 'sent', 'input'):
        if src in ds.column_names and 'sentence' not in ds.column_names:
            ds = ds.rename_column(src, 'sentence')
    missing = {'sentence', 'labels'} - set(ds.column_names)
    if missing:
        raise RuntimeError(f'dataset is missing {missing}; '
                           f'columns present: {ds.column_names}')
    ds = ds.remove_columns([c for c in ds.column_names
                            if c not in ('sentence', 'labels')])

    # stratified splitting requires labels to be a ClassLabel feature
    from datasets import ClassLabel
    if not isinstance(ds.features['labels'], ClassLabel):
        if ds.features['labels'].dtype == 'string':
            ds = ds.map(lambda b: {'labels': [LABEL_NAMES.index(x.lower())
                                              for x in b['labels']]}, batched=True)
        ds = ds.cast_column('labels', ClassLabel(names=LABEL_NAMES))
    print('label distribution:',
          {LABEL_NAMES[i]: ds['labels'].count(i) for i in range(3)})

    # PhraseBank ships as a single split, so carve out train / val / test
    ds = ds.train_test_split(test_size=0.3, seed=seed, stratify_by_column='labels')
    tmp = ds['test'].train_test_split(test_size=0.5, seed=seed,
                                      stratify_by_column='labels')
    splits = {'train': ds['train'], 'val': tmp['train'], 'test': tmp['test']}

    def tok(b):
        return tokenizer(b['sentence'], truncation=True, max_length=MAX_LEN)

    for k in splits:
        splits[k] = splits[k].map(tok, batched=True,
                                  remove_columns=['sentence'])
    return splits


# ---------------------------------------------------------------------------
# Model construction per method
# ---------------------------------------------------------------------------

def host_ram_gb():
    """Resident set size of this process, in GB. Colab kills the session at ~12 GB."""
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**30
    except Exception:                                             # noqa: BLE001
        return 0.0


def build_model(method, rank=8, alpha=16, seed=42):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, dtype=torch.float32)
    model.config.pad_token_id = model.config.eos_token_id

    if method == 'head':
        for n, p in model.named_parameters():
            p.requires_grad = 'score' in n

    elif method == 'full':
        for p in model.parameters():
            p.requires_grad = True

    elif method == 'lora':
        from peft import LoraConfig, TaskType, get_peft_model
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=alpha,
                         lora_dropout=0.05, bias='none',
                         target_modules=['q_proj', 'v_proj'],
                         modules_to_save=['score'])
        model = get_peft_model(model, cfg)

    else:
        raise ValueError(method)

    return model.to(DEVICE)


def count_trainable(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    # Exclude the freshly-initialised classification head from the adapter budget
    head = sum(p.numel() for n, p in model.named_parameters()
               if 'score' in n and p.requires_grad)
    return tr - head, head, tot


# ---------------------------------------------------------------------------
# Train / evaluate
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, gold = [], []
    for batch in loader:
        labels = batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(**batch).logits
        preds.append(logits.float().argmax(-1).cpu())
        gold.append(labels)
    p = torch.cat(preds).numpy()
    g = torch.cat(gold).numpy()
    return accuracy_score(g, p), f1_score(g, p, average='macro')


def run(method, rank, lr, seed, splits, collator):
    model = build_model(method, rank=rank, seed=seed)
    n_adapter, n_head, n_total = count_trainable(model)

    tr = DataLoader(splits['train'], batch_size=BATCH, shuffle=True,
                    collate_fn=collator)
    va = DataLoader(splits['val'], batch_size=64, collate_fn=collator)
    te = DataLoader(splits['test'], batch_size=64, collate_fn=collator)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = EPOCHS * len(tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps,
                                                pct_start=0.1)

    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    t0 = time.time()
    for _ in range(EPOCHS):
        model.train()
        for batch in tr:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            # Unscale before clipping, otherwise the norm is computed on the
            # scaled gradients and the clip threshold means nothing.
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
    secs = time.time() - t0

    va_acc, va_f1 = evaluate(model, va)
    te_acc, te_f1 = evaluate(model, te)
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == 'cuda' else 0.0

    # Tear the run down completely. `del model` alone is NOT enough: the
    # optimizer holds references to every parameter (and, for full fine-tuning,
    # two AdamW moment buffers per parameter), and the scheduler holds the
    # optimizer. Miss any one of them and a ~2 GB fp32 copy of the model stays
    # reachable, so 20 runs exhaust the 12 GB Colab gives you.
    del model, params, opt, sched, scaler, tr, va, te
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    return dict(method=method, rank=rank, lr=lr, seed=seed,
                adapter_params=n_adapter, head_params=n_head, total_params=n_total,
                pct=100 * n_adapter / n_total, val_acc=va_acc, val_f1=va_f1,
                test_acc=te_acc, test_f1=te_f1, seconds=secs, peak_gb=peak,
                host_ram_gb=host_ram_gb())


# ---------------------------------------------------------------------------
# Main grid
# ---------------------------------------------------------------------------

RESULTS_FILE = 'colab_results.json'

# (method, rank, candidate learning rates)
# Ranks 1 / 4 / 16 span two orders of magnitude of adapter capacity, which is
# enough to see the shape of the rank curve. Dropping r=2 and r=8 costs 8 runs.
GRID = [
    ('head', 0, [1e-3, 3e-3]),
    ('lora', 1, [1e-4, 3e-4]),
    ('lora', 4, [1e-4, 3e-4]),
    ('lora', 16, [1e-4, 3e-4]),
    ('full', 0, [5e-6, 2e-5]),
]


def load_records():
    try:
        return json.load(open(RESULTS_FILE))
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_records(records):
    json.dump(records, open(RESULTS_FILE, 'w'), indent=1)


def done(records, method, rank, lr, seed):
    return any(r['method'] == method and r['rank'] == rank
               and abs(r['lr'] - lr) < 1e-12 and r['seed'] == seed
               for r in records)


def main(only=None):
    """Run the grid. Safe to re-run: finished jobs are skipped, not repeated.

    `only` filters by method, e.g. main(only='lora') or main(only=['head', 'full']).
    """
    if isinstance(only, str):
        only = [only]

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    splits = get_data(tokenizer)
    collator = DataCollatorWithPadding(tokenizer)
    print({k: len(v) for k, v in splits.items()})

    records = load_records()
    if records:
        print(f'resuming: {len(records)} finished runs already in {RESULTS_FILE}')

    for method, rank, lrs in GRID:
        if only and method not in only:
            continue
        tag = method + (f'-r{rank}' if method == 'lora' else '')

        # 1) pick the learning rate on VALIDATION using one seed
        for lr in lrs:
            if done(records, method, rank, lr, SEEDS[0]):
                continue
            r = run(method, rank, lr, SEEDS[0], splits, collator)
            records.append(r)
            save_records(records)
            print(f'    {tag} lr={lr:.0e} seed={SEEDS[0]} done | '
                  f'{r["seconds"]:.0f}s | RAM {r["host_ram_gb"]:.1f} GB')

        probes = [r for r in records if r['method'] == method and r['rank'] == rank
                  and r['seed'] == SEEDS[0]]
        best_lr = max(probes, key=lambda r: r['val_f1'])['lr']

        # 2) repeat the winning setting on the remaining seeds
        for seed in SEEDS[1:]:
            if done(records, method, rank, best_lr, seed):
                continue
            r = run(method, rank, best_lr, seed, splits, collator)
            records.append(r)
            save_records(records)
            print(f'    {tag} lr={best_lr:.0e} seed={seed} done | '
                  f'{r["seconds"]:.0f}s | RAM {r["host_ram_gb"]:.1f} GB')

        # Also dump the raw records into the cell output. If Colab recycles the
        # runtime, /content is wiped and colab_results.json goes with it - but
        # the notebook output survives, so the numbers can still be recovered.
        print(f'  [{tag}] complete (lr* = {best_lr:.0e})')
        print('RECOVERY_JSON ' + json.dumps(records, separators=(',', ':')))

    print_table(records)


def print_table(records=None):
    """Print whatever results exist so far. Safe to call at any point."""
    records = records if records is not None else load_records()
    if not records:
        print('no results yet')
        return
    print('\n' + '=' * 96)
    print('RESULTS TABLE  (paste this back into the chat)')
    print('=' * 96)
    print(f'{"method":<12}{"adapter":>12}{"% total":>9}{"lr*":>9}'
          f'{"test acc":>18}{"macro-F1":>18}{"sec":>8}{"GB":>7}{"n":>3}')
    print('-' * 96)
    for method, rank, _ in GRID:
        tag = method + (f'-r{rank}' if method == 'lora' else '')
        rs = [r for r in records if r['method'] == method and r['rank'] == rank]
        if not rs:
            print(f'{tag:<12}{"(not run yet)":>20}')
            continue
        best_lr = max(rs, key=lambda r: r['val_f1'])['lr']
        rs = [r for r in rs if r['lr'] == best_lr]
        a = np.array([r['test_acc'] for r in rs])
        f = np.array([r['test_f1'] for r in rs])
        r0 = rs[0]
        sa = a.std(ddof=1) if len(a) > 1 else 0.0
        sf = f.std(ddof=1) if len(f) > 1 else 0.0
        print(f'{tag:<12}{r0["adapter_params"]:>12,}{r0["pct"]:>9.3f}{best_lr:>9.1e}'
              f'{a.mean():>12.4f} +/-{sa:<5.4f}{f.mean():>12.4f} +/-{sf:<5.4f}'
              f'{np.mean([r["seconds"] for r in rs]):>8.0f}'
              f'{max(r["peak_gb"] for r in rs):>7.1f}{len(rs):>3}')
    print('-' * 96)
    print(f'model {MODEL_NAME} | {records[0]["total_params"]:,} total parameters | '
          f'{EPOCHS} epochs | seeds {SEEDS}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile study2_data_scaling.py
"""
Study 2 - Is LoRA's low rank acting as regularisation in low-resource financial NLP?
====================================================================================
Motivated by an unresolved confound in Study 1. There, rank mattered on a 558k
synthetic setup and did not on Qwen2.5-0.5B - but model size, corpus and training-set
size all changed at once, so the effect could not be attributed to any of them.

This study holds the model (Qwen2.5-0.5B) and the corpus (Financial PhraseBank)
fixed and varies ONLY the amount of labelled training data.

    H1  low-rank adaptation is more beneficial the scarcer the supervision
    H2  rank sensitivity decreases as training data grows
    H3  the benefit is specific to the low-rank structure, not to merely having
        fewer trainable parameters   (tested by the `fullreg` control arm)

Design decisions that the conclusions depend on:

  * TRAIN is subsampled; VAL and TEST stay at full size for every run, so
    evaluation noise does not grow as the training set shrinks.
  * Subsampling is stratified. PhraseBank is 62% neutral and 12% negative; an
    unstratified draw of 100 would leave ~12 negatives and the n=100 condition
    would be measuring class imbalance rather than data scarcity.
  * Budget: EPOCHS are held constant (see BUDGET_MODE below). Because
    steps = epochs x n / batch, holding epochs constant necessarily lets the
    number of gradient updates vary with n, and holding steps constant
    necessarily lets the number of exposures per example vary. The two cannot
    both be fixed. Equal epochs is the primary setting, on the grounds that it
    lets each condition actually use its full training set and matches how a
    practitioner with n labels would really train; the fixed-step variant is
    retained as a robustness check, and the qualitative conclusion should
    survive both.
  * The generalisation gap (train accuracy - test accuracy) is recorded, because
    "LoRA wins when data is scarce" is consistent with better optimisation as
    well as with regularisation. Only the gap distinguishes the two.
  * `fullreg` = full fine-tuning with heavy weight decay. If LoRA's advantage is
    just capacity control, a capacity-controlled full fine-tune should match it.

Usage:
    python study2_data_scaling.py                 # whole grid, resumable
    python study2_data_scaling.py --only lora     # one arm at a time
    python study2_data_scaling.py --sizes 100 250 # one slice
"""

import argparse
import gc
import json
import math
import os
import time

import numpy as np
import torch
from datasets import ClassLabel
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding)

# Reuse the audited data loading from Study 1.
from peft_study import (LABEL_NAMES, MODEL_NAME, MAX_LEN, AMP_DTYPE, USE_AMP,
                        DEVICE, load_phrasebank, host_ram_gb)

RESULTS_FILE = 'study2_results.json'

BATCH = 16

# --- training budget -------------------------------------------------------
# steps = epochs x n / batch, so with batch and n fixed you cannot hold BOTH
# the number of updates and the number of exposures per example constant. The
# two modes trade one confound for the other:
#
#   'epochs' (primary, per supervisor's request)
#       every example is seen EPOCHS times in every condition. Comparable
#       exposure per example; unequal number of gradient updates (n=100 gets
#       50 updates, n=2417 gets 1,209).
#
#   'steps'  (secondary, robustness check)
#       every condition gets the same number of gradient updates. Comparable
#       optimisation budget; unequal exposure per example (n=100 sees each
#       example 48 times, n=2417 sees each twice).
#
# Neither is "correct". Running the small grid under both and checking that the
# qualitative conclusion survives is what actually settles it.
BUDGET_MODE = 'epochs'
# 5 epochs: Study 1 converged on the full training set within 3 epochs, so 5
# plus early stopping leaves headroom while keeping the grid inside a few hours
# of single-GPU time.
EPOCHS = 5               # used when BUDGET_MODE == 'epochs'
FIXED_STEPS = 300        # used when BUDGET_MODE == 'steps'
STEP_CEILING = 2000      # safety cap only; should not bind at these sizes

# Floor on the number of gradient updates, in 'epochs' mode only.
#
# Measured, not assumed. At n=100, 5 epochs is 5 x ceil(100/16) = 35 updates, and
# 35 updates train nothing: a pilot run reached 0.486 test accuracy with 0.54
# train accuracy - below the 0.62 majority-class baseline, and still improving
# when the budget ran out. The same condition at 150 steps reached 0.718 test
# with 1.000 train accuracy and a 0.282 generalisation gap.
#
# Without the floor the small-n columns would measure update starvation rather
# than data scarcity, and H1 could not be tested at all. The floor binds only at
# n=100 and n=250; from n=500 up, the epoch budget already exceeds it, so equal
# epochs is preserved wherever it is meaningful. The cost is that those two
# conditions see each example more often than the larger ones - recorded in
# `exposures_per_example` and reported.
MIN_STEPS = 150

# Validation checkpoints are spaced so that EVERY run gets the same NUMBER of
# them, rather than the same interval in steps. A fixed interval would give the
# n=100 condition 2 checkpoints and the n=2417 condition 48, so early stopping
# would be far more sensitive at large n than at small n - and the evaluation
# overhead would grow without bound (at n=2417 it exceeded the training time).
EVALS_PER_RUN = 12
PATIENCE = 4             # early stop after this many evals without val-F1 gain
SEEDS = [42, 43, 44]
TRAIN_SIZES = [100, 250, 500, 1000, 2417]     # 2417 = the full training split

# method, rank, weight_decay, learning rate
ARMS = [
    ('lora', 1, 0.01, 3e-4),
    ('lora', 2, 0.01, 3e-4),
    ('lora', 4, 0.01, 3e-4),
    ('lora', 8, 0.01, 3e-4),
    ('lora', 16, 0.01, 3e-4),
    ('full', 0, 0.01, 5e-6),
    ('fullreg', 0, 0.50, 5e-6),   # H3 control: capacity-limited full fine-tuning
]


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def build_splits(tokenizer, seed=0):
    """Full-size val/test, plus the full training pool to subsample from."""
    ds = load_phrasebank()
    if 'label' in ds.column_names and 'labels' not in ds.column_names:
        ds = ds.rename_column('label', 'labels')
    if not isinstance(ds.features['labels'], ClassLabel):
        ds = ds.cast_column('labels', ClassLabel(names=LABEL_NAMES))

    ds = ds.train_test_split(test_size=0.3, seed=seed, stratify_by_column='labels')
    tmp = ds['test'].train_test_split(test_size=0.5, seed=seed,
                                      stratify_by_column='labels')
    splits = {'train': ds['train'], 'val': tmp['train'], 'test': tmp['test']}

    def tok(b):
        return tokenizer(b['sentence'], truncation=True, max_length=MAX_LEN)

    for k in splits:
        splits[k] = splits[k].map(tok, batched=True, remove_columns=['sentence'])
    return splits


def stratified_subsample(train, n, seed):
    """Draw n examples keeping the class proportions of the full training pool."""
    if n >= len(train):
        return train
    labels = np.array(train['labels'])
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(labels, return_counts=True)
    quota = np.floor(counts / counts.sum() * n).astype(int)
    quota = np.maximum(quota, 1)                    # never drop a class entirely
    while quota.sum() > n:                          # trim the largest class first
        quota[np.argmax(quota)] -= 1
    while quota.sum() < n:
        quota[np.argmax(counts - quota)] += 1

    idx = []
    for c, q in zip(classes, quota):
        pool = np.flatnonzero(labels == c)
        idx.extend(rng.choice(pool, size=min(q, len(pool)), replace=False).tolist())
    rng.shuffle(idx)
    return train.select(idx)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

def build_model(method, rank, seed):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, dtype=torch.float32)
    model.config.pad_token_id = model.config.eos_token_id

    if method in ('full', 'fullreg'):
        for p in model.parameters():
            p.requires_grad = True
    elif method == 'lora':
        from peft import LoraConfig, TaskType, get_peft_model
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=2 * rank,
                         lora_dropout=0.05, bias='none',
                         target_modules=['q_proj', 'v_proj'],
                         modules_to_save=['score'])
        model = get_peft_model(model, cfg)
    else:
        raise ValueError(method)
    return model.to(DEVICE)


def count_trainable(model):
    """(adapter params, classification-head params, total).

    The head is randomly initialised and therefore trained under every arm, so
    it is reported separately and excluded from the adapter budget.
    """
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    head = sum(p.numel() for n, p in model.named_parameters()
               if 'score' in n and p.requires_grad)
    return tr - head, head, tot


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, gold = [], []
    for batch in loader:
        labels = batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(**batch).logits
        preds.append(logits.float().argmax(-1).cpu())
        gold.append(labels)
    p, g = torch.cat(preds).numpy(), torch.cat(gold).numpy()
    return accuracy_score(g, p), f1_score(g, p, average='macro')


# ---------------------------------------------------------------------------
# One run
# ---------------------------------------------------------------------------

def budget_for(n_examples):
    """Total optimisation steps for a training set of this size."""
    per_epoch = math.ceil(n_examples / BATCH)
    if BUDGET_MODE == 'epochs':
        steps = max(EPOCHS * per_epoch, MIN_STEPS)
    elif BUDGET_MODE == 'steps':
        steps = FIXED_STEPS
    else:
        raise ValueError(BUDGET_MODE)
    return min(steps, STEP_CEILING), per_epoch


def run(method, rank, wd, lr, n_train, seed, splits, collator):
    train = stratified_subsample(splits['train'], n_train, seed)
    model = build_model(method, rank, seed)
    n_adapter, n_head, n_total = count_trainable(model)
    max_steps, steps_per_epoch = budget_for(len(train))
    eval_every = max(1, max_steps // EVALS_PER_RUN)

    tr = DataLoader(train, batch_size=BATCH, shuffle=True, collate_fn=collator,
                    drop_last=False)
    tr_eval = DataLoader(train, batch_size=64, collate_fn=collator)
    va = DataLoader(splits['val'], batch_size=64, collate_fn=collator)
    te = DataLoader(splits['test'], batch_size=64, collate_fn=collator)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
                                                total_steps=max_steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    best = {'val_f1': -1.0, 'val_acc': 0.0, 'step': 0}
    since_improve, step = 0, 0
    t0 = time.time()

    while step < max_steps:
        model.train()
        for batch in tr:
            if step >= max_steps:
                break
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1

            if step % eval_every == 0 or step == max_steps:
                # Only VALIDATION is evaluated during training. Touching the
                # test set at every checkpoint would triple the evaluation cost
                # (a forward pass over 518 examples of a 0.5B model is ~4 s) and
                # buys nothing: early stopping decides when to stop, and the
                # reported model is the one training stopped at.
                va_acc, va_f1 = evaluate(model, va)
                if va_f1 > best['val_f1']:
                    best = {'val_acc': va_acc, 'val_f1': va_f1, 'step': step}
                    since_improve = 0
                else:
                    since_improve += 1
                model.train()
        if since_improve >= PATIENCE:
            break

    # Final-state metrics: the generalisation gap must be read at the END of
    # training, not at the early-stopped optimum, or it is not a gap at all.
    fin_tr_acc, _ = evaluate(model, tr_eval)
    fin_te_acc, fin_te_f1 = evaluate(model, te)
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == 'cuda' else 0.0

    del model, params, opt, sched, scaler, tr, tr_eval, va, te
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    return dict(method=method, rank=rank, wd=wd, lr=lr, n_train=len(train),
                n_requested=n_train, seed=seed,
                adapter_params=n_adapter, head_params=n_head, total_params=n_total,
                pct=100 * n_adapter / n_total,
                budget_mode=BUDGET_MODE, max_steps=max_steps,
                floor_applied=bool(BUDGET_MODE == 'epochs'
                                   and EPOCHS * steps_per_epoch < MIN_STEPS),
                steps_per_epoch=steps_per_epoch, eval_every=eval_every,
                n_evals=step // eval_every,
                epochs_run=step / steps_per_epoch,
                exposures_per_example=step * BATCH / len(train),
                best_val_acc=best['val_acc'], best_val_f1=best['val_f1'],
                best_step=best['step'], steps_run=step,
                # Reported at the point training stopped, which is the early-
                # stopped model. train/test are measured on the same state, so
                # the gap between them is meaningful.
                test_acc=fin_te_acc, test_f1=fin_te_f1,
                train_acc=fin_tr_acc,
                gap=fin_tr_acc - fin_te_acc,
                seconds=secs, peak_gb=peak, host_ram_gb=host_ram_gb())


# ---------------------------------------------------------------------------
# Grid
# ---------------------------------------------------------------------------

def load_records():
    try:
        return json.load(open(RESULTS_FILE))
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def done(recs, method, rank, n, seed):
    # budget_mode is part of the identity of a run: the same arm under the two
    # budgets is two different experiments and must not be treated as a repeat.
    return any(r['method'] == method and r['rank'] == rank
               and r['n_requested'] == n and r['seed'] == seed
               and r.get('budget_mode', 'steps') == BUDGET_MODE for r in recs)


def tag_of(method, rank):
    return f'lora-r{rank}' if method == 'lora' else method


def main(only=None, sizes=None, budget=None):
    global BUDGET_MODE
    if budget:
        BUDGET_MODE = budget
    if isinstance(only, str):
        only = [only]
    sizes = sizes or TRAIN_SIZES

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    splits = build_splits(tokenizer)
    collator = DataCollatorWithPadding(tokenizer)
    budget_desc = (f'{EPOCHS} epochs' if BUDGET_MODE == 'epochs'
                   else f'{FIXED_STEPS} steps')
    print({k: len(v) for k, v in splits.items()},
          f'| budget: {budget_desc} ({BUDGET_MODE} mode), patience {PATIENCE}')
    if BUDGET_MODE == 'epochs':
        print('  steps by size:',
              {n: budget_for(min(n, len(splits['train'])))[0] for n in (sizes or TRAIN_SIZES)})

    recs = load_records()
    if recs:
        print(f'resuming: {len(recs)} runs already done')

    todo = [(m, rk, wd, lr, n, s)
            for n in sizes
            for (m, rk, wd, lr) in ARMS
            for s in SEEDS
            if (not only or m in only) and not done(recs, m, rk, n, s)]
    print(f'{len(todo)} runs to go\n')

    for i, (m, rk, wd, lr, n, s) in enumerate(todo, 1):
        r = run(m, rk, wd, lr, n, s, splits, collator)
        recs.append(r)
        json.dump(recs, open(RESULTS_FILE, 'w'), indent=1)
        print(f'  [{i}/{len(todo)}] {tag_of(m, rk):<10} n={r["n_train"]:<5} '
              f'seed={s} | test {r["test_acc"]:.4f} | gap {r["gap"]:+.4f} | '
              f'{r["seconds"]:.0f}s | RAM {r["host_ram_gb"]:.1f} GB')
        if i % 6 == 0 or i == len(todo):
            print('RECOVERY_JSON ' + json.dumps(recs, separators=(',', ':')))

    summarize(recs)


def summarize(recs=None):
    recs = recs if recs is not None else load_records()
    if not recs:
        print('no results yet')
        return
    tags = [tag_of(m, rk) for (m, rk, _, _) in ARMS]
    sizes = sorted({r['n_requested'] for r in recs})

    for metric, label in [('test_acc', 'TEST ACCURACY'), ('gap', 'GENERALISATION GAP')]:
        print(f'\n{"=" * 78}\n{label}  (mean over seeds)\n{"=" * 78}')
        print(f'{"method":<11}' + ''.join(f'{"n=" + str(n):>13}' for n in sizes))
        print('-' * 78)
        for tag in tags:
            row = f'{tag:<11}'
            for n in sizes:
                sel = [r[metric] for r in recs
                       if tag_of(r['method'], r['rank']) == tag and r['n_requested'] == n]
                row += f'{np.mean(sel):>13.4f}' if sel else f'{"-":>13}'
            print(row)
    print()


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--only', nargs='*', default=None)
    ap.add_argument('--sizes', nargs='*', type=int, default=None)
    ap.add_argument('--summarize', action='store_true')
    a = ap.parse_args()
    if a.summarize:
        summarize()
    else:
        main(only=a.only, sizes=a.sizes)


In [ ]:
import study2_data_scaling as s2
caps = {n: max(s2.EPOCHS * -(-n // s2.BATCH), s2.MIN_STEPS) for n in s2.TRAIN_SIZES}
print("budget:", s2.EPOCHS, "epochs, floor", s2.MIN_STEPS, "steps | patience", s2.PATIENCE)
print("step cap by size :", caps)
print("exposures/example:", {n: round(v * s2.BATCH / n, 1) for n, v in caps.items()})
print("total runs:", len(s2.TRAIN_SIZES) * len(s2.ARMS) * len(s2.SEEDS))


## 3. Save results to Google Drive

**Do this before walking away.** A runtime recycle wipes `/content` and every
completed run with it; Drive survives. The authorisation popup is why this has to
happen while you are still at the machine.

This also clears the two pilot runs from the smoke test — one of them used the
old 35-step budget and would otherwise be mixed in with the real grid.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json
os.makedirs("/content/drive/MyDrive/llm_study2", exist_ok=True)
s2.RESULTS_FILE = "/content/drive/MyDrive/llm_study2/study2_results.json"

# start clean: the pilot runs used a different budget and must not be reused
json.dump([], open(s2.RESULTS_FILE, "w"))
if os.path.exists("study2_results.json"):
    os.remove("study2_results.json")
print("results file:", s2.RESULTS_FILE)
print("runs recorded:", len(s2.load_records()), "(expected 0)")


## 4. Full run  (~3 hours, unattended)

The whole grid in one cell. Results are written after **every** run, so an
interruption costs at most the run in progress. If it dies, re-run this cell —
finished runs are skipped.

Watch the first few lines before you leave: `gap` should be positive and `RAM`
flat around 3–5 GB.

In [ ]:
import time, datetime

t0 = time.time()
print("started", datetime.datetime.now().strftime("%H:%M:%S"),
      "| already done:", len(s2.load_records()))
try:
    s2.main()
except KeyboardInterrupt:
    print("\ninterrupted by user")
except Exception as e:
    print(f"\n!!! stopped: {type(e).__name__}: {e}")
    print("results are saved — re-run this cell to resume")
finally:
    done = len(s2.load_records())
    total = len(s2.TRAIN_SIZES) * len(s2.ARMS) * len(s2.SEEDS)
    print(f"\nfinished {datetime.datetime.now().strftime('%H:%M:%S')}"
          f" | {(time.time()-t0)/60:.0f} min | {done}/{total} runs")


## 5. Summary — paste both tables into the chat

In [ ]:
s2.summarize()


## 6. Optional — robustness check under the fixed-step budget

Equal epochs and equal steps cannot both be held (steps = epochs × n / batch).
This re-runs the small sizes under the fixed-step budget; if the qualitative
conclusion is unchanged, the choice of budget did not drive it. ~15 min.

In [ ]:
s2.main(sizes=[100, 250, 500], budget="steps")


## 7. If something goes wrong

| Symptom | What to do |
|---|---|
| `ImportError: ... torchao` | Re-run §1, restart runtime, then §2 |
| Session crashed, RAM exhausted | Re-run §4 — it resumes. Send me the `RAM` figures |
| Runtime recycled | Re-run §2, then §3 **but skip the `json.dump([])` line**, then §4 |
| Drive was never mounted | Send me the last `RECOVERY_JSON` line from the dead output |

`s2.summarize()` prints whatever finished — partial results are still useful.